## Pulls live batting data

In [2]:
pip install MLB-StatsAPI requests pandas beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install MLB-StatsAPI requests pandas beautifulsoup4 statsapi

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement statsapi (from versions: none)
ERROR: No matching distribution found for statsapi


In [4]:
pip install mlb-statsapi pandas requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import statsapi
from datetime import datetime
import time
import random

class BatterData:
    def __init__(self):
        self.cache = {}
        self.cache_timeout = 3600  # 1 hour

    def get_player_stats(self, player_name: str) -> dict:
        """Get player information and current season stats"""
        try:
            # Look up player
            player_lookup = statsapi.lookup_player(player_name)
            
            if not player_lookup:
                print(f"Could not find player: {player_name}")
                return None
            
            player = player_lookup[0]
            player_id = player['id']
            
            # Get team info
            team_id = player.get('currentTeam', {}).get('id')
            team_name = "Free Agent"  # Default if no team found
            
            if team_id:
                team_info = statsapi.lookup_team(team_id)
                if team_info:
                    team_name = team_info[0].get('name', "Unknown Team")

            # Get current season stats
            stats_request = statsapi.player_stat_data(player_id, group="hitting", type="season")
            
            # Initialize player data
            player_data = {
                'id': player_id,
                'name': player['fullName'],
                'position': player['primaryPosition']['abbreviation'],
                'team': team_name,
                'stats': {
                    'avg': 0.250,  # Default values that will be overwritten
                    'slg': 0.400,
                    'ops': 0.700,
                    'pitch_stats': {
                        'fastball': {'avg': 0.250, 'slg': 0.400},
                        'curveball': {'avg': 0.250, 'slg': 0.400},
                        'slider': {'avg': 0.250, 'slg': 0.400},
                        'changeup': {'avg': 0.250, 'slg': 0.400}
                    }
                }
            }

            # Update with actual stats if available
            if stats_request and isinstance(stats_request, dict) and 'stats' in stats_request:
                try:
                    current_stats = stats_request['stats'][0]['stats']
                    player_data['stats'].update({
                        'avg': float(current_stats['avg']),
                        'slg': float(current_stats['slg']),
                        'ops': float(current_stats['ops']),
                    })

                    # Calculate pitch-specific stats based on overall performance
                    avg = player_data['stats']['avg']
                    slg = player_data['stats']['slg']
                    
                    player_data['stats']['pitch_stats'] = {
                        'fastball': {'avg': round(avg * 1.1, 3), 'slg': round(slg * 1.1, 3)},
                        'curveball': {'avg': round(avg * 0.9, 3), 'slg': round(slg * 0.9, 3)},
                        'slider': {'avg': round(avg * 0.85, 3), 'slg': round(slg * 0.85, 3)},
                        'changeup': {'avg': round(avg * 0.95, 3), 'slg': round(slg * 0.95, 3)}
                    }
                    
                    # Print retrieved stats for verification
                    print(f"\nRetrieved stats for {player_data['name']}:")
                    print(f"AVG: {player_data['stats']['avg']}")
                    print(f"SLG: {player_data['stats']['slg']}")
                    print(f"OPS: {player_data['stats']['ops']}")
                    
                except (KeyError, IndexError, TypeError) as e:
                    print(f"Warning: Could not parse some stats: {e}")
                    # Keep default values if stats parsing fails

            # Cache the data
            self.cache[player_id] = {
                'timestamp': time.time(),
                'data': player_data
            }
            
            return player_data
                
        except Exception as e:
            print(f"Error getting player stats: {e}")
            return None

class PitchingStrategy:
    def __init__(self):
        self.batter_data = BatterData()
        self.game_tracker = GameTracker()
        self.current_batter = None
        
    def set_current_batter(self, batter_name: str) -> None:
        """Set current batter and fetch their stats"""
        player_data = self.batter_data.get_player_stats(batter_name)
        
        if player_data:
            self.current_batter = player_data
            print(f"\nCurrent Batter: {player_data['name']}")
            print(f"Team: {player_data['team']}")
            print(f"Position: {player_data['position']}")
            print(f"AVG: {player_data['stats']['avg']:.3f}")
            print(f"SLG: {player_data['stats']['slg']:.3f}")
            print(f"OPS: {player_data['stats']['ops']:.3f}")
        else:
            print(f"Using league average stats for {batter_name}")
            self.current_batter = {
                'name': batter_name,
                'team': 'Unknown',
                'position': 'Unknown',
                'stats': {
                    'avg': 0.250,
                    'slg': 0.400,
                    'ops': 0.700,
                    'pitch_stats': {
                        'fastball': {'avg': 0.250, 'slg': 0.400},
                        'curveball': {'avg': 0.250, 'slg': 0.400},
                        'slider': {'avg': 0.250, 'slg': 0.400},
                        'changeup': {'avg': 0.250, 'slg': 0.400}
                    }
                }
            }

    def calculate_pitch_score(self, pitch_type: str, game_situation: dict) -> float:
        """Calculate score for each pitch type"""
        score = 0.0
        
        # Batter stats factor
        stats = self.current_batter['stats']['pitch_stats'][pitch_type]
        score -= stats['avg'] * 50  # Reduced weight of AVG
        score -= stats['slg'] * 25  # Reduced weight of SLG
        
        # Count leverage adjustments
        balls = game_situation['balls']
        strikes = game_situation['strikes']
        
        # Fastball situations
        if balls == 3 or (strikes == 2 and balls < 3):
            score += 30 if pitch_type == 'fastball' else 0  # Need strikes
            
        # Breaking ball situations
        if strikes < 2 and balls < 3:
            if strikes == 0:
                score += 20 if pitch_type in ['curveball', 'slider'] else 0  # First pitch breaking ball
            else:
                score += 15 if pitch_type in ['slider', 'changeup'] else 0  # Off-speed when ahead
        
        # Runner situation adjustments
        runners = game_situation['runners']
        if any(runners):
            if runners[2]:  # Runner on third
                score += 20 if pitch_type in ['fastball', 'slider'] else 0  # Better control
            if runners[1]:  # Runner on second
                score += 15 if pitch_type in ['curveball', 'changeup'] else 0  # Harder to hit
            if runners[0]:  # Runner on first
                score += 10 if pitch_type in ['fastball', 'slider'] else 0  # Better for double play
        
        # Game situation adjustments
        inning = game_situation['inning']
        score_diff = game_situation.get('score_diff', 0)
        
        # Late game adjustments
        if inning >= 7:
            if abs(score_diff) <= 2:  # Close game
                score += 15 if pitch_type in ['fastball', 'slider'] else 0  # Better control
            else:  # Not close game
                score += 10 if pitch_type in ['changeup', 'curveball'] else 0  # Mix it up
        
        # Add some randomization to prevent always choosing the same pitch
        score += random.uniform(-5, 5)
        
        return score

    def recommend_pitch(self, game_situation: dict) -> str:
        """Recommend best pitch for situation"""
        pitch_scores = {}
        for pitch_type in ['fastball', 'curveball', 'slider', 'changeup']:
            pitch_scores[pitch_type] = self.calculate_pitch_score(pitch_type, game_situation)
        
        # Print scores for debugging
        print("\nPitch Scores:")
        for pitch, score in pitch_scores.items():
            print(f"{pitch}: {score:.2f}")
        
        return max(pitch_scores.items(), key=lambda x: x[1])[0]

class GameTracker:
    def __init__(self):
        self.game_log = []
        
    def add_pitch(self, batter: str, pitch_type: str, result: str, situation: dict):
        """Record pitch in game log"""
        self.game_log.append({
            'timestamp': datetime.now(),
            'batter': batter,
            'pitch_type': pitch_type,
            'result': result,
            'situation': situation
        })
    
    def get_summary(self) -> dict:
        """Get summary of pitches thrown"""
        pitch_counts = {}
        for pitch in self.game_log:
            pitch_type = pitch['pitch_type']
            pitch_counts[pitch_type] = pitch_counts.get(pitch_type, 0) + 1
        return pitch_counts

def get_game_situation() -> dict:
    """Get current game situation from user"""
    print("\nEnter game situation:")
    try:
        return {
            'balls': int(input("Balls (0-3): ")),
            'strikes': int(input("Strikes (0-2): ")),
            'outs': int(input("Outs (0-2): ")),
            'inning': int(input("Inning (1-9): ")),
            'runners': [
                input("Runner on first? (y/n): ").lower() == 'y',
                input("Runner on second? (y/n): ").lower() == 'y',
                input("Runner on third? (y/n): ").lower() == 'y'
            ],
            'score_diff': int(input("Score difference (+ for ahead, - for behind): "))
        }
    except ValueError:
        print("Invalid input. Please enter numeric values.")
        return get_game_situation()

def main():
    strategy = PitchingStrategy()
    
    while True:
        # Get batter
        batter_name = input("\nEnter batter name (or 'quit' to end): ")
        if batter_name.lower() == 'quit':
            break
            
        strategy.set_current_batter(batter_name)
        
        while True:
            # Get situation and recommendation
            situation = get_game_situation()
            pitch = strategy.recommend_pitch(situation)
            print(f"\nRecommended pitch: {pitch}")
            
            # Record result
            result = input("Enter result (hit/out/ball/strike/foul): ").lower()
            strategy.game_tracker.add_pitch(
                strategy.current_batter['name'], 
                pitch, 
                result, 
                situation
            )
            
            # Continue with same batter?
            if input("\nNext batter? (y/n): ").lower() == 'y':
                break
    
    # Show game summary
    print("\nGame Summary:")
    summary = strategy.game_tracker.get_summary()
    total_pitches = sum(summary.values())
    
    if total_pitches > 0:
        for pitch_type, count in summary.items():
            percentage = (count / total_pitches) * 100
            print(f"{pitch_type}: {count} ({percentage:.1f}%)")
    else:
        print("No pitches thrown")

if __name__ == "__main__":
    main()


## With Pitching

In [26]:
import statsapi
from datetime import datetime
import time
import random

class BatterData:
    def __init__(self):
        self.cache = {}
        self.cache_timeout = 3600  # 1 hour

    def get_player_stats(self, player_name: str) -> dict:
        """Get player information and current season stats"""
        try:
            # Look up player
            player_lookup = statsapi.lookup_player(player_name)
            
            if not player_lookup:
                print(f"Could not find player: {player_name}")
                return None
            
            player = player_lookup[0]
            player_id = player['id']
            
            # Get team info
            team_id = player.get('currentTeam', {}).get('id')
            team_name = "Free Agent"  # Default if no team found
            
            if team_id:
                team_info = statsapi.lookup_team(team_id)
                if team_info:
                    team_name = team_info[0].get('name', "Unknown Team")

            # Get current season stats
            stats_request = statsapi.player_stat_data(player_id, group="hitting", type="season")
            
            # Initialize player data
            player_data = {
                'id': player_id,
                'name': player['fullName'],
                'position': player['primaryPosition']['abbreviation'],
                'team': team_name,
                'stats': {
                    'avg': 0.250,  # Default values that will be overwritten
                    'slg': 0.400,
                    'ops': 0.700,
                    'pitch_stats': {
                        'fastball': {'avg': 0.250, 'slg': 0.400},
                        'curveball': {'avg': 0.250, 'slg': 0.400},
                        'slider': {'avg': 0.250, 'slg': 0.400},
                        'changeup': {'avg': 0.250, 'slg': 0.400}
                    }
                }
            }

            # Update with actual stats if available
            if stats_request and isinstance(stats_request, dict) and 'stats' in stats_request:
                try:
                    current_stats = stats_request['stats'][0]['stats']
                    player_data['stats'].update({
                        'avg': float(current_stats['avg']),
                        'slg': float(current_stats['slg']),
                        'ops': float(current_stats['ops']),
                    })

                    # Calculate pitch-specific stats based on overall performance
                    avg = player_data['stats']['avg']
                    slg = player_data['stats']['slg']
                    
                    player_data['stats']['pitch_stats'] = {
                        'fastball': {'avg': round(avg * 1.1, 3), 'slg': round(slg * 1.1, 3)},
                        'curveball': {'avg': round(avg * 0.9, 3), 'slg': round(slg * 0.9, 3)},
                        'slider': {'avg': round(avg * 0.85, 3), 'slg': round(slg * 0.85, 3)},
                        'changeup': {'avg': round(avg * 0.95, 3), 'slg': round(slg * 0.95, 3)}
                    }
                    
                    # Print retrieved stats for verification
                    print(f"\nRetrieved stats for {player_data['name']}:")
                    print(f"AVG: {player_data['stats']['avg']}")
                    print(f"SLG: {player_data['stats']['slg']}")
                    print(f"OPS: {player_data['stats']['ops']}")
                    
                except (KeyError, IndexError, TypeError) as e:
                    print(f"Warning: Could not parse some stats: {e}")
                    # Keep default values if stats parsing fails

            # Cache the data
            self.cache[player_id] = {
                'timestamp': time.time(),
                'data': player_data
            }
            
            return player_data
                
        except Exception as e:
            print(f"Error getting player stats: {e}")
            return None

class PitcherData:
    def __init__(self):
        self.cache = {}
        self.cache_timeout = 3600  # 1 hour

    def get_pitcher_stats(self, pitcher_name: str) -> dict:
        """Get pitcher's current season stats and pitch-specific data"""
        try:
            pitcher_lookup = statsapi.lookup_player(pitcher_name)
            
            if not pitcher_lookup:
                print(f"Could not find pitcher: {pitcher_name}")
                return None
            
            pitcher = pitcher_lookup[0]
            pitcher_id = pitcher['id']
            
            # Get team info
            team_id = pitcher.get('currentTeam', {}).get('id')
            team_name = "Free Agent"
            
            if team_id:
                team_info = statsapi.lookup_team(team_id)
                if team_info:
                    team_name = team_info[0].get('name', "Unknown Team")

            # Get current season stats
            stats_request = statsapi.player_stat_data(pitcher_id, group="pitching", type="season")
            
            # Initialize pitcher data
            pitcher_data = {
                'id': pitcher_id,
                'name': pitcher['fullName'],
                'team': team_name,
                'throws': pitcher.get('pitchHand', {}).get('code', 'R'),
                'stats': {
                    'era': 4.50,
                    'whip': 1.30,
                    'k_per_9': 8.0,
                    'pitch_stats': {
                        'fastball': {
                            'velocity': 93.0,
                            'usage': 0.50,
                            'batting_avg': 0.250,
                            'strike_pct': 0.65
                        },
                        'curveball': {
                            'velocity': 78.0,
                            'usage': 0.15,
                            'batting_avg': 0.200,
                            'strike_pct': 0.60
                        },
                        'slider': {
                            'velocity': 84.0,
                            'usage': 0.20,
                            'batting_avg': 0.190,
                            'strike_pct': 0.62
                        },
                        'changeup': {
                            'velocity': 85.0,
                            'usage': 0.15,
                            'batting_avg': 0.220,
                            'strike_pct': 0.58
                        }
                    }
                },
                'fatigue': {
                    'pitch_count': 0,
                    'max_pitches': 100,
                    'fatigue_factor': 1.0
                }
            }

            # Update with actual stats if available
            if stats_request and isinstance(stats_request, dict) and 'stats' in stats_request:
                try:
                    current_stats = stats_request['stats'][0]['stats']
                    pitcher_data['stats'].update({
                        'era': float(current_stats.get('era', 4.50)),
                        'whip': float(current_stats.get('whip', 1.30)),
                        'k_per_9': float(current_stats.get('strikeoutsPer9Inn', 8.0)),
                    })
                    
                except (KeyError, IndexError, TypeError) as e:
                    print(f"Warning: Could not parse some pitcher stats: {e}")

            return pitcher_data
                
        except Exception as e:
            print(f"Error getting pitcher stats: {e}")
            return None

    def update_fatigue(self, pitcher_data: dict, pitches_thrown: int) -> dict:
        """Update pitcher fatigue based on pitch count"""
        pitcher_data['fatigue']['pitch_count'] += pitches_thrown
        pitch_count = pitcher_data['fatigue']['pitch_count']
        max_pitches = pitcher_data['fatigue']['max_pitches']
        fatigue_factor = 1.0 + (pitch_count / max_pitches) * 0.5
        pitcher_data['fatigue']['fatigue_factor'] = fatigue_factor
        return pitcher_data

class PitchingStrategy:
    def __init__(self):
        self.batter_data = BatterData()
        self.pitcher_data = PitcherData()
        self.game_tracker = GameTracker()
        self.current_batter = None
        self.current_pitcher = None

    def set_current_batter(self, batter_name: str) -> None:
        """Set current batter and fetch their stats"""
        player_data = self.batter_data.get_player_stats(batter_name)
        
        if player_data:
            self.current_batter = player_data
            print(f"\nCurrent Batter: {player_data['name']}")
            print(f"Team: {player_data['team']}")
            print(f"Position: {player_data['position']}")
            print(f"AVG: {player_data['stats']['avg']:.3f}")
            print(f"SLG: {player_data['stats']['slg']:.3f}")
            print(f"OPS: {player_data['stats']['ops']:.3f}")
        else:
            print(f"Using league average stats for {batter_name}")
            self.current_batter = self._get_default_batter(batter_name)

    def set_current_pitcher(self, pitcher_name: str) -> None:
        """Set current pitcher and fetch their stats"""
        pitcher_data = self.pitcher_data.get_pitcher_stats(pitcher_name)
        
        if pitcher_data:
            self.current_pitcher = pitcher_data
            print(f"\nCurrent Pitcher: {pitcher_data['name']}")
            print(f"Team: {pitcher_data['team']}")
            print(f"Throws: {pitcher_data['throws']}")
            print(f"ERA: {pitcher_data['stats']['era']:.2f}")
            print(f"WHIP: {pitcher_data['stats']['whip']:.2f}")
            print(f"K/9: {pitcher_data['stats']['k_per_9']:.1f}")
        else:
            print(f"Using league average stats for {pitcher_name}")
            self.current_pitcher = self._get_default_pitcher(pitcher_name)


    def calculate_pitch_score(self, pitch_type: str, game_situation: dict) -> float:
        """Calculate score for each pitch type considering both batter and pitcher stats"""
        score = 0.0
        
        # Get relevant stats
        batter_stats = self.current_batter['stats']['pitch_stats'][pitch_type]
        pitcher_stats = self.current_pitcher['stats']['pitch_stats'][pitch_type]
        fatigue = self.current_pitcher['fatigue']['fatigue_factor']
        
        # Batter factors
        score -= batter_stats['avg'] * 50
        score -= batter_stats['slg'] * 25
        
        # Pitcher factors
        score += (1 - pitcher_stats['batting_avg']) * 40  # Better BAA = higher score
        score += pitcher_stats['strike_pct'] * 30         # Better strike% = higher score
        score -= (fatigue - 1.0) * 20                     # Fatigue penalty
        
        # Count leverage adjustments
        balls = game_situation['balls']
        strikes = game_situation['strikes']
        
        # Must-throw-strike situations
        if balls == 3 or (strikes == 2 and balls < 3):
            score += 30 * pitcher_stats['strike_pct']  # Heavily favor high strike% pitches
            
        # Ahead in count - can waste a pitch
        if strikes > balls and strikes < 2:
            score += 15 if pitch_type in ['slider', 'curveball'] else 0
        
        # Runner situation adjustments
        runners = game_situation['runners']
        if any(runners):
            if runners[2]:  # Runner on third
                score += 20 * pitcher_stats['strike_pct']  # Need control
            if runners[1]:  # Runner on second
                score += 15 if pitcher_stats['batting_avg'] < 0.200 else 0  # Need weak contact
            if runners[0]:  # Runner on first
                score += 10 if pitch_type in ['fastball', 'slider'] else 0  # Better for double play
        
        # Game situation adjustments
        inning = game_situation['inning']
        score_diff = game_situation.get('score_diff', 0)
        
        # Late game adjustments
        if inning >= 7:
            if abs(score_diff) <= 2:  # Close game
                score += 15 if pitch_type in ['fastball', 'slider'] else 0  # Better control
            else:  # Not close game
                score += 10 if pitch_type in ['changeup', 'curveball'] else 0  # Mix it up
        
        # Fatigue considerations
        if fatigue > 1.2:  # Significant fatigue
            score -= 10 if pitch_type == 'fastball' else 0  # Harder to maintain velocity
        
        # Add randomization
        score += random.uniform(-3, 3)
        
        return score

    def recommend_pitch(self, game_situation: dict) -> str:
        """Recommend best pitch based on all factors"""
        pitch_scores = {}
        
        # Get previous pitch if available
        prev_pitch = None
        if self.game_tracker.game_log:
            prev_pitch = self.game_tracker.game_log[-1]['pitch_type']
        
        for pitch_type in ['fastball', 'curveball', 'slider', 'changeup']:
            base_score = self.calculate_pitch_score(pitch_type, game_situation)
            if pitch_type == prev_pitch:
                base_score *= 0.9  # Penalty for repeating pitches
            pitch_scores[pitch_type] = base_score
        
        # Print detailed pitch analysis
        print("\nPitch Analysis:")
        for pitch, score in pitch_scores.items():
            velocity = self.current_pitcher['stats']['pitch_stats'][pitch]['velocity']
            strike_pct = self.current_pitcher['stats']['pitch_stats'][pitch]['strike_pct']
            print(f"{pitch}: {score:.2f} (Vel: {velocity}, Strike%: {strike_pct:.3f})")
        
        return max(pitch_scores.items(), key=lambda x: x[1])[0]

    def _get_default_batter(self, name: str) -> dict:
        """Return default batter stats"""
        return {
            'name': name,
            'team': 'Unknown',
            'position': 'Unknown',
            'stats': {
                'avg': 0.250,
                'slg': 0.400,
                'ops': 0.700,
                'pitch_stats': {
                    'fastball': {'avg': 0.250, 'slg': 0.400},
                    'curveball': {'avg': 0.250, 'slg': 0.400},
                    'slider': {'avg': 0.250, 'slg': 0.400},
                    'changeup': {'avg': 0.250, 'slg': 0.400}
                }
            }
        }

    def _get_default_pitcher(self, name: str) -> dict:
        """Return default pitcher stats"""
        return {
            'name': name,
            'team': 'Unknown',
            'throws': 'R',
            'stats': {
                'era': 4.50,
                'whip': 1.30,
                'k_per_9': 8.0,
                'pitch_stats': {
                    'fastball': {'velocity': 93.0, 'usage': 0.50, 'batting_avg': 0.250, 'strike_pct': 0.65},
                    'curveball': {'velocity': 78.0, 'usage': 0.15, 'batting_avg': 0.200, 'strike_pct': 0.60},
                    'slider': {'velocity': 84.0, 'usage': 0.20, 'batting_avg': 0.190, 'strike_pct': 0.62},
                    'changeup': {'velocity': 85.0, 'usage': 0.15, 'batting_avg': 0.220, 'strike_pct': 0.58}
                }
            },
            'fatigue': {
                'pitch_count': 0,
                'max_pitches': 100,
                'fatigue_factor': 1.0
            }
        }

class GameTracker:
    def __init__(self):
        self.game_log = []
        self.pitcher_stats = {}
        
    def add_pitch(self, pitcher: str, batter: str, pitch_type: str, result: str, situation: dict):
        """Record pitch in game log with enhanced tracking"""
        pitch_data = {
            'timestamp': datetime.now(),
            'pitcher': pitcher,
            'batter': batter,
            'pitch_type': pitch_type,
            'result': result,
            'situation': situation
        }
        self.game_log.append(pitch_data)
        
        # Initialize pitcher stats if needed
        if pitcher not in self.pitcher_stats:
            self.pitcher_stats[pitcher] = {
                'total_pitches': 0,
                'strikes': 0,
                'balls': 0,
                'hits': 0,
                'outs': 0,
                'pitch_types': {},
                'results': {}
            }
        
        # Update pitch counts
        self.pitcher_stats[pitcher]['total_pitches'] += 1
        if result in ['strike', 'foul', 'out']:
            self.pitcher_stats[pitcher]['strikes'] += 1
        elif result == 'ball':
            self.pitcher_stats[pitcher]['balls'] += 1
        elif result == 'hit':
            self.pitcher_stats[pitcher]['hits'] += 1
        if result == 'out':
            self.pitcher_stats[pitcher]['outs'] += 1
            
        # Track pitch type usage
        if pitch_type not in self.pitcher_stats[pitcher]['pitch_types']:
            self.pitcher_stats[pitcher]['pitch_types'][pitch_type] = 0
        self.pitcher_stats[pitcher]['pitch_types'][pitch_type] += 1
        
        # Track results
        if result not in self.pitcher_stats[pitcher]['results']:
            self.pitcher_stats[pitcher]['results'][result] = 0
        self.pitcher_stats[pitcher]['results'][result] += 1

    def get_summary(self) -> dict:
        """Get detailed game summary"""
        return {
            'total_pitches': len(self.game_log),
            'by_pitcher': self.pitcher_stats
        }

def print_game_summary(summary: dict):
    """Print detailed game summary"""
    print("\n=== Game Summary ===")
    print(f"Total Pitches: {summary['total_pitches']}")
    
    for pitcher, stats in summary['by_pitcher'].items():
        print(f"\nPitcher: {pitcher}")
        print(f"Total Pitches: {stats['total_pitches']}")
        
        # Calculate strike percentage
        strike_pct = (stats['strikes'] / stats['total_pitches'] * 100) if stats['total_pitches'] > 0 else 0
        print(f"Strike %: {strike_pct:.1f}%")
        
        # Pitch type distribution
        print("\nPitch Distribution:")
        for pitch_type, count in stats['pitch_types'].items():
            percentage = (count / stats['total_pitches'] * 100)
            print(f"{pitch_type}: {count} ({percentage:.1f}%)")
        
        # Results breakdown
        print("\nResults Breakdown:")
        for result, count in stats['results'].items():
            percentage = (count / stats['total_pitches'] * 100)
            print(f"{result}: {count} ({percentage:.1f}%)")

def main():
    strategy = PitchingStrategy()
    
    # Initial pitcher setup
    pitcher_name = input("Enter starting pitcher name: ")
    strategy.set_current_pitcher(pitcher_name)
    
    while True:
        # Get batter
        batter_name = input("\nEnter batter name (or 'quit' to end): ")
        if batter_name.lower() == 'quit':
            break
        
        # Option to change pitcher
        change_pitcher = input("Change pitcher? (y/n): ").lower() == 'y'
        if change_pitcher:
            pitcher_name = input("Enter new pitcher name: ")
            strategy.set_current_pitcher(pitcher_name)
            
        strategy.set_current_batter(batter_name)
        
        while True:
            # Get situation and recommendation
            situation = get_game_situation()
            
            # Update pitcher fatigue
            strategy.pitcher_data.update_fatigue(
                strategy.current_pitcher,
                strategy.game_tracker.pitcher_stats.get(pitcher_name, {}).get('total_pitches', 0)
            )
            
            # Get pitch recommendation
            pitch = strategy.recommend_pitch(situation)
            print(f"\nRecommended pitch: {pitch}")
            
            # Record result
            result = input("Enter result (hit/out/ball/strike/foul): ").lower()
            strategy.game_tracker.add_pitch(
                strategy.current_pitcher['name'],
                strategy.current_batter['name'],
                pitch,
                result,
                situation
            )
            
            # Show current pitcher stats
            pitcher_stats = strategy.game_tracker.pitcher_stats.get(strategy.current_pitcher['name'], {})
            if pitcher_stats:
                print(f"\nCurrent Pitcher Status:")
                print(f"Total Pitches: {pitcher_stats.get('total_pitches', 0)}")
                total_pitches = pitcher_stats.get('total_pitches', 0)
                strikes = pitcher_stats.get('strikes', 0)
                if total_pitches > 0:
                    print(f"Strike %: {(strikes / total_pitches * 100):.1f}%")
                
                # Check fatigue
                fatigue = strategy.current_pitcher['fatigue']['fatigue_factor']
                print(f"Fatigue Factor: {fatigue:.2f}")
                
                if fatigue > 1.5:
                    print("Warning: Pitcher showing significant fatigue")
            
            # Continue with same batter?
            if input("\nNext batter? (y/n): ").lower() == 'y':
                break
    
    # Show final game summary
    print_game_summary(strategy.game_tracker.get_summary())

if __name__ == "__main__":
    main()


KeyboardInterrupt: Interrupted by user